In [ ]:
import pdfplumber
import pandas as pd
import re

def extract_hdb_rpi_fixed(file_path):
    all_text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            all_text += page.extract_text() + "\n"

    # finds 4-digit years from 1990 to 2025
    years = re.findall(r'\b(199\d|20[0-2]\d)\b', all_text)
    years = sorted(list(set(years)), reverse=True)
    
    records = []
    
    # split the text into sections by year
    for i in range(len(years)):
        year = years[i]
        start_marker = year
        # end marker is the next year in the list, or the end of the text
        end_marker = years[i+1] if i+1 < len(years) else "$"
        
        # extract the chunk of text belonging to this year
        pattern = f"{start_marker}(.*?)(?={end_marker})"
        section = re.search(pattern, all_text, re.DOTALL)
        
        if section:
            section_text = section.group(1)
            
            # find Quarters and Indexes in this specific section
            matches = re.findall(r'\b(IV|III|II|I|11|1)\b\s+(\d{2,3}\.\d)\b', section_text)
            
            for q_raw, index_val in matches:
                q_map = {'1':'Q1', 'I':'Q1', '11':'Q2', 'II':'Q2', 'III':'Q3', 'IV':'Q4'}
                records.append({
                    'Period': f"{year}-{q_map[q_raw]}",
                    'Index': float(index_val)
                })

    df = pd.DataFrame(records)
    
    # sort and clean
    df = df.drop_duplicates(subset=['Period']).sort_values('Period', ascending=False).reset_index(drop=True)
    return df

df_rpi = extract_hdb_rpi_fixed("../../data/raw/4Q2025 RPI Table.pdf")
print(df_rpi.head(10))

    Period  Index
0  2025-Q4  203.6
1  2025-Q3  203.7
2  2025-Q2  202.9
3  2025-Q1  201.0
4  2024-Q4  180.4
5  2024-Q3  178.5
6  2024-Q2  187.9
7  2024-Q1  183.7
8  2023-Q4  171.9
9  2023-Q3  168.1


In [11]:
# seperate function to extract the year and quarter from the "Period" column
df_rpi['year'] = df_rpi['Period'].apply(lambda x: x.split('-')[0])
df_rpi['quarter'] = df_rpi['Period'].apply(lambda x: x.split('-')[1])

# remove Q from quarter
df_rpi['quarter'] = df_rpi['quarter'].str.replace('Q', '')

# drop period
df_rpi = df_rpi.drop(columns=['Period'])
df_rpi.head()

,Index,year,quarter
0,203.6,2025,4
1,203.7,2025,3
2,202.9,2025,2
3,201.0,2025,1
4,180.4,2024,4


In [12]:
# save to csv
df_rpi.to_csv("../../data/cleaned/rpi_cleaned.csv", index=False)